# 1. Preprocessing: skull stripping, orientation alignment and brain-masked histogram matching

The BraTS-PEDs-v1 volumes all share the same header (LPS orientation, 240×240×155 voxels, 1 mm), but the images are not skull-stripped and intensities are not calibrated across scanners and sites. This notebook prepares the data for segmentation:

1. **Skull stripping** with SynthStrip gives a brain mask per subject.
2. **Orientation alignment.** Some heads are rotated inside the shared grid (one by ~90°), which makes the brain masks fit poorly. Subjects are rigidly registered to a template subject, and rotated subjects are resampled.
3. **Brain-masked histogram matching.** Each sequence is matched to a reference subject using only voxels inside the brain mask, so zero background and skull/scalp do not influence the matching.

Heavy steps are also available as command-line scripts (`scripts/`). This notebook calls the same functions and shows the QC figures. Cells have not been executed in the published version; the numbers quoted in the text come from the author's run on BraTS-PEDs-v1.

In [ ]:
import os
from pathlib import Path

# Run from the repository root so the relative paths in the configuration resolve.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print(f"Working directory: {Path.cwd()}")

from IPython.display import Image, display

from brats_peds_seg import alignment
from brats_peds_seg.histogram_matching import run_histogram_matching

ORIGINAL_ROOT = Path("data/BraTS-PEDs-v1")                       # Training/ and Validation/ subject folders
ALIGNED_ROOT = Path("data/BraTS-PEDs-v1_rigid_aligned")
HISTMATCHED_ROOT = Path("data/BraTS-PEDs-v1_rigid_aligned_histmatched")
REPORTS = Path("reports")

## 1.1 Skull stripping

SynthStrip runs in Docker (`docker/synthstrip/`). The command below writes `{case}-t1n_mask.nii.gz` next to each subject's t1n; the mask is applied to all four co-registered sequences. Run it once in a terminal:

```bash
scripts/skullstrip_dataset.sh data/BraTS-PEDs-v1/Training data/BraTS-PEDs-v1/Validation
```

## 1.2 Orientation screen

Rigid registration with mutual information could not reliably tell a truly rotated head from an upright one: with several metrics, masks and multi-template consensus it repeatedly preferred wrong 90°/180° flips for upright heads. Gross rotations are therefore identified visually from contact sheets (mid-sagittal and mid-axial t1n slice of every subject); registration then only does the fine alignment.

In BraTS-PEDs-v1 every subject is upright with the face anterior except **BraTS-PED-00042-000**, which is tipped ~90° about the left–right axis.

In [ ]:
pages = alignment.orientation_screen(ORIGINAL_ROOT, REPORTS / "orientation_screen")
display(Image(filename=str(pages[0])))

## 1.3 Template choice

Every subject is registered (identity start) to a provisional subject to estimate its orientation. A good template has a typical orientation (small median rotation to all other subjects), a small tumor that lies inside the brain mask, and near-isotropic resolution (`sharpness_ratio` ≥ 0.8; thick-slice scans fall well below). Candidates are ranked automatically and the final choice is made after viewing them: BraTS-PED-00037-000 was used: it is sharp, has no visible mass on the mid slices, and is within 0.25° of the most typical orientation; the highest-ranked sharp candidate (BraTS-PED-00002-000) has a large thalamic mass.

In [ ]:
candidates = alignment.rank_template_candidates(ORIGINAL_ROOT, REPORTS / "template_selection", "BraTS-PED-00190-000")
display(candidates.round(3))

In [ ]:
TEMPLATE = "BraTS-PED-00037-000"
GROSS_ROTATION_STARTS = {"BraTS-PED-00042-000": ((-90, 0, 0),)}  # from the orientation screen
ROTATION_THRESHOLD_DEG = 10.0

## 1.4 Rigid alignment

Every t1n is registered to the template (6-DOF rigid, Mattes mutual information on percentile-clipped intensities inside a head mask, multi-resolution). The transform is applied to all four sequences (linear interpolation) and to the segmentation (nearest neighbour) on the same grid. Subjects rotated by more than the threshold are resampled; all others are hard-linked unchanged, so they are not interpolated for no reason.

In the author's run, 8 subjects exceeded 10°: BraTS-PED-00042-000 (87°) and seven tilts of 11–17°.

In [ ]:
result = alignment.align_dataset(ORIGINAL_ROOT, ALIGNED_ROOT, TEMPLATE, GROSS_ROTATION_STARTS, ROTATION_THRESHOLD_DEG)
display(result.loc[result["realigned"], ["case_id", "split", "rotation_deg", "translation_mm", "start_rotation_deg"]].round(1))
display(Image(filename=str(ALIGNED_ROOT / "reports" / "rotation_histogram.png")))
for page in sorted((ALIGNED_ROOT / "reports" / "qc").glob("*.png")):
    display(Image(filename=str(page)))

## 1.5 New brain masks for resampled subjects and tumor coverage

Resampled subjects get new SynthStrip masks. The check below counts tumor voxels outside the brain mask (Training subjects have labels). Rotated heads were the cause of the worst cases: for BraTS-PED-00042-000 the tumor outside the mask fell from 51 % to 0.01 %. The remaining cases are upright subjects whose tumor lies at the brain edge.

In [ ]:
realigned = result.loc[result["realigned"]]
alignment.run_synthstrip([ALIGNED_ROOT / s / c for c, s in zip(realigned["case_id"], realigned["split"])],
                         Path("docker/synthstrip/run_synthstrip.sh"))
coverage = alignment.tumor_mask_coverage(ORIGINAL_ROOT, ALIGNED_ROOT)
coverage.to_csv(ALIGNED_ROOT / "reports" / "tumor_mask_coverage.csv", index=False)
for name in ["original", "aligned"]:
    values = coverage[f"tumor_outside_pct_{name}"]
    print(f"{name:>8}: >1% tumor outside mask: {(values > 1).sum()}, >5%: {(values > 5).sum()}, max {values.max():.1f}%")

## 1.6 Brain-masked histogram matching

Each scan's in-mask intensity distribution is matched to the same sequence of a reference subject: the Training subject closest to the robust centre of the in-mask mean, standard deviation, skewness and kurtosis. Voxels outside the mask are set to 0.

In [ ]:
output_root, summary = run_histogram_matching(ALIGNED_ROOT, HISTMATCHED_ROOT)
display(summary.round(3))
display(Image(filename=str(output_root / "reports" / "example_histograms_before_after.png")))
display(Image(filename=str(output_root / "reports" / "before_after_statistics.png")))

## 1.7 Summary

- The aligned dataset (`data/BraTS-PEDs-v1_rigid_aligned`) holds images, brain masks, labels, per-subject rigid transforms and QC reports.
- The histogram-matched dataset holds brain-masked matched images and masks; labels are always read from the aligned dataset.
- Labels obtained later for the Validation subjects must be transformed with the saved transform of any realigned subject before scoring.
- Limitation: tumor tissue outside the SynthStrip mask is zero in brain-masked inputs and cannot be segmented from them.